# Phase 2 – Data Collection & Preparation (Telco Customer Churn)

This notebook implements **Phase 2: Data Collection & Preparation** for the Telco Customer Churn dataset.

We will:

1. Load and validate the raw dataset  
2. Perform data preprocessing (missing values, duplicates, outliers, data types, formatting)  
3. Encode categorical variables and scale numeric features  
4. Save both **cleaned** and **model-ready** datasets

---

## 1. Setup and Library Imports

In this section, we import all the Python libraries needed for data loading, exploration, and preprocessing.


In [ ]:
# Standard data libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Preprocessing utilities
from sklearn.preprocessing import StandardScaler

# Display options (optional)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Data Collection – Loading the Raw Dataset

The dataset file should be placed in the same folder as this notebook.

- **File name:** `WA_Fn-UseC_-Telco-Customer-Churn.csv`  
- **Access method:** Local CSV read using `pandas.read_csv()`

In [ ]:
# Path to the raw data file (assumes same directory as the notebook)
DATA_PATH = "datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the raw dataset
df_raw = pd.read_csv(DATA_PATH)

# Make a working copy so we keep the original untouched
df = df_raw.copy()

df_raw.head()


## 3. Data Acquisition Validation

Here we confirm that the dataset:

- Can be read without errors  
- Has the expected structure (rows, columns)  
- Has no obvious corruption issues (e.g., wrong delimiter, broken encoding)

We will inspect:

- Shape (rows × columns)  
- Column names  
- Data types and non-null counts  
- Sample records  
- Basic descriptive statistics

In [ ]:
# Shape of the dataset
print("Shape (rows, columns):", df.shape)

# Column names
print("\nColumn names:")
print(df.columns.tolist())

# Info: data types and non-null counts
print("\nDataFrame info:")
df.info()

# Preview first 5 rows
print("\nHead:")
display(df.head())

# Basic descriptive statistics for numeric columns
print("\nDescriptive statistics (numeric):")
display(df.describe())

# Descriptive statistics for object (categorical) columns
print("\nDescriptive statistics (categorical):")
display(df.describe(include="object"))


## 4. Missing Data Handling

Steps:

1. **Identify missing values** in each column.  
2. Decide on an appropriate strategy (drop vs. impute).  
3. Apply imputation for relevant columns.

From experience with this dataset, `TotalCharges` may contain blank strings that become `NaN` when converted to numeric. We will:

- Convert `TotalCharges` to numeric with `errors="coerce"` so invalid entries become `NaN`.  
- Impute missing `TotalCharges` using the **median** value.


In [ ]:
# 4.1 Check missing values before any transformations
print("Missing values per column (before):")
print(df.isna().sum())

# 4.2 Strip whitespace from column names to avoid subtle issues
df.columns = df.columns.str.strip()

# 4.3 Convert 'TotalCharges' to numeric
# Some entries may be empty strings; errors='coerce' turns them into NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("\nMissing values per column (after converting TotalCharges):")
print(df.isna().sum())

# 4.4 Impute missing TotalCharges with the median
median_total_charges = df["TotalCharges"].median()
df["TotalCharges"] = df["TotalCharges"].fillna(median_total_charges)

print("\nMissing values per column (after imputing TotalCharges):")
print(df.isna().sum())


## 5. Duplicate Removal

We check for duplicate customer records and remove them if present.


In [ ]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# Drop duplicate rows if any exist
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")

print("New shape after duplicate removal:", df.shape)


## 6. Outlier Detection & Treatment

Outliers can distort statistical analyses and model training. Here we:

1. Select numeric columns  
2. Use the **Interquartile Range (IQR)** method to flag outliers  
3. Decide to **remove** rows with extreme outliers for this phase

> Note: In a real project, you might **cap**, **transform**, or carefully investigate outliers instead of always removing them.


In [ ]:
# Select numeric columns for outlier analysis
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)

# Compute IQR-based bounds for each numeric column
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("\nLower bounds:")
print(lower_bound)
print("\nUpper bounds:")
print(upper_bound)

# Create an outlier mask
outlier_mask = ((df[numeric_cols] < lower_bound) | (df[numeric_cols] > upper_bound)).any(axis=1)
print("\nNumber of rows flagged as outliers:", outlier_mask.sum())

# Option: Keep a version without outliers
df_no_outliers = df.loc[~outlier_mask].copy()
print("Shape after removing outliers:", df_no_outliers.shape)


In [ ]:
# Optional: Visualize distributions before/after outlier removal for a key column

column_to_plot = "MonthlyCharges"

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.boxplot(df[column_to_plot].dropna())
plt.title("With Outliers")

plt.subplot(1, 2, 2)
plt.boxplot(df_no_outliers[column_to_plot].dropna())
plt.title("Without Outliers")

plt.suptitle(f"Outlier Effect on '{column_to_plot}'")
plt.tight_layout()
plt.show()


## 7. Data Type Correction

We ensure that each column has an appropriate data type:

- `customerID`: string  
- `TotalCharges`, `MonthlyCharges`, `tenure`: numeric  
- `SeniorCitizen`: numeric but logically categorical (0/1)  
- Other service columns: categorical (object / category)


In [ ]:
# Work with the no-outliers version from this point onward
df_clean = df_no_outliers.copy()

print("Data types before adjustments:")
print(df_clean.dtypes)

# Ensure 'customerID' is treated as string
df_clean["customerID"] = df_clean["customerID"].astype(str)

# Confirm numerical columns are numeric
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Treat SeniorCitizen as categorical (0/1)
df_clean["SeniorCitizen"] = df_clean["SeniorCitizen"].astype("int64")

print("\nData types after adjustments:")
print(df_clean.dtypes)


## 8. Formatting & Standardization

We clean up formatting inconsistencies, such as:

- Trimming whitespace in string values  
- Converting categorical text to a consistent case (e.g., lowercase)

This helps avoid duplicated categories like `"Yes"`, `" yes"`, `"YES"`.


In [ ]:
# Strip whitespace and standardize case for all object (string) columns
object_cols = df_clean.select_dtypes(include="object").columns.tolist()
print("Object columns:", object_cols)

for col in object_cols:
    df_clean[col] = df_clean[col].str.strip()          # remove leading/trailing spaces
    df_clean[col] = df_clean[col].str.lower()          # convert to lowercase for consistency

# Check unique values for a few categorical columns
for col in ["gender", "partner", "dependents", "phone service" if "phone service" in df_clean.columns else "phoneservice", "churn"]:
    if col in df_clean.columns:
        print(f"\nUnique values in '{col}':")
        print(df_clean[col].unique())


## 9. Categorical Data Encoding

To prepare the data for machine learning models, we encode categorical variables numerically.

Steps:

1. Separate the **target variable** (`churn`) from the features.  
2. Encode `churn` as binary (1 = churn, 0 = no churn).  
3. Apply **one-hot encoding** (`pd.get_dummies`) to the remaining categorical features.


In [ ]:
# --- Defensive fix for missing 'churn' key ---

# Identify the exact column name that represents churn
print("Available columns:", df_clean.columns.tolist())

# Find the actual churn column (case-insensitive match)
churn_col = None
for c in df_clean.columns:
    if c.strip().lower() == "churn":
        churn_col = c
        break

if churn_col is None:
    raise KeyError("No 'Churn' column found in dataset. Check dataset headers.")

# Standardize the column name
df_clean.rename(columns={churn_col: "churn"}, inplace=True)

# Now proceed as before
target_col = "churn"
y = df_clean[target_col].map({"yes": 1, "no": 0})
print("Target distribution:")
print(y.value_counts())

# Drop non-feature columns and target
feature_df = df_clean.drop(columns=["customerID", target_col])

# One-hot encode
X = pd.get_dummies(feature_df, drop_first=True)

print("\nFeature matrix shape after encoding:", X.shape)
X.head()

## 10. Feature Scaling / Normalization

Numeric features may be on very different scales. Many machine learning algorithms perform better when features are **standardized**.

We will:

1. Identify numeric columns in `X`  
2. Apply `StandardScaler` to transform them to zero mean and unit variance  
3. Create a new scaled feature DataFrame with the same column names


In [ ]:
# Identify numeric feature columns
numeric_feature_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print("Number of numeric feature columns:", len(numeric_feature_cols))

# Initialize the scaler
scaler = StandardScaler()

# Fit and transform numeric columns
X_scaled_array = scaler.fit_transform(X[numeric_feature_cols])

# Create a DataFrame with scaled values
X_scaled = pd.DataFrame(X_scaled_array, columns=numeric_feature_cols, index=X.index)

# Final model-ready DataFrame: target + scaled features
model_df = pd.concat([y.rename("churn"), X_scaled], axis=1)
print("Model-ready dataframe shape:", model_df.shape)
model_df.head()


## 11. Saving Raw, Cleaned, and Model-Ready Datasets

For reproducibility and future analysis, we save:

- **Raw dataset** (as originally loaded)  
- **Cleaned dataset** (after preprocessing, before encoding/scaling)  
- **Model-ready dataset** (encoded & scaled features + target)

These files can be used in later phases (e.g., modeling & evaluation).


In [ ]:
# File names for outputs
RAW_OUT_PATH = "cleaned_data/telco_churn_raw.csv"
CLEAN_OUT_PATH = "cleaned_data/telco_churn_clean.csv"
MODEL_OUT_PATH = "cleaned_data/telco_churn_model_ready.csv"

# Save datasets
df_raw.to_csv(RAW_OUT_PATH, index=False)
df_clean.to_csv(CLEAN_OUT_PATH, index=False)
model_df.to_csv(MODEL_OUT_PATH, index=False)

print("Saved:")
print(" -", RAW_OUT_PATH)
print(" -", CLEAN_OUT_PATH)
print(" -", MODEL_OUT_PATH)


## 12. Preprocessing Summary (Documentation)

**Data Source**  
- Local CSV: `WA_Fn-UseC_-Telco-Customer-Churn.csv`  
- Represents telecom customers and their churn status.

**Key Preprocessing Steps**  

1. **Data Acquisition Validation**
   - Verified shape, column names, and types.
   - Inspected sample rows and descriptive statistics.

2. **Missing Data Handling**
   - Identified missing values (especially in `TotalCharges` after numeric conversion).
   - Imputed `TotalCharges` using the **median**.

3. **Duplicate Removal**
   - Checked for complete-row duplicates and removed any found.

4. **Outlier Detection & Treatment**
   - Used the IQR method on numeric features.
   - Removed rows flagged as extreme outliers to create `df_no_outliers`.

5. **Data Type Correction**
   - Converted `TotalCharges`, `MonthlyCharges`, and `tenure` to numeric.
   - Ensured `customerID` is string and `SeniorCitizen` is integer (0/1).

6. **Formatting & Standardization**
   - Stripped whitespace from all string columns.
   - Converted categorical text to lowercase to avoid duplicate categories.

7. **Categorical Encoding**
   - Encoded target `churn` as binary (1 = yes, 0 = no).
   - Applied one-hot encoding (`pd.get_dummies`) to features.

8. **Feature Scaling**
   - Standardized numeric features with `StandardScaler`.

9. **Saved Outputs**
   - `telco_churn_raw.csv` – Raw data as originally loaded.
   - `telco_churn_clean.csv` – Cleaned/preprocessed dataset.
   - `telco_churn_model_ready.csv` – Model-ready dataset (encoded + scaled).

This completes **Phase 2 – Data Collection & Preparation** for the Telco Customer Churn project.
